In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DNS_FILE = Path(
    "../data/raw/CTU-13/capture20110810.binetflow"
)

columns = [
    "StartTime",
    "Dur",
    "Proto",
    "SrcAddr",
    "Sport",
    "DstAddr",
    "Dport",
    "TotPkts",
    "TotBytes",
    "SrcBytes",
    "Label"
]

dns_df = pd.read_csv(
    DNS_FILE,
    usecols=columns
)

dns_df["StartTime"] = pd.to_datetime(
    dns_df["StartTime"],
    errors="coerce"
)

dns_df["Dport_numeric"] = pd.to_numeric(
    dns_df["Dport"].astype(str).str.strip(),
    errors="coerce"
)

dns_df = dns_df.dropna(
    subset=[
        "StartTime",
        "SrcAddr",
        "DstAddr"
    ]
)

print("DNS analysis dataframe:", dns_df.shape)

DNS analysis dataframe: (2824636, 12)


In [2]:
dns_flows = dns_df[
    dns_df["Dport_numeric"] == 53
].copy()

print("DNS destination-port-53 flows:", len(dns_flows))

print("\nProtocols:")
print(
    dns_flows["Proto"].value_counts()
)

print("\nTop sources:")
display(
    dns_flows["SrcAddr"]
    .value_counts()
    .head(20)
)

DNS destination-port-53 flows: 992650

Protocols:
Proto
udp    992648
tcp         2
Name: count, dtype: int64

Top sources:


SrcAddr
147.32.84.138    529464
147.32.84.59     109163
147.32.85.25      44859
147.32.85.34      35001
147.32.84.165     29197
147.32.86.20      15304
147.32.84.170     12266
147.32.85.7        8553
147.32.84.212      7718
147.32.86.99       7374
147.32.85.23       5791
147.32.84.164      5778
147.32.86.182      5620
147.32.84.131      5557
147.32.85.89       5436
147.32.85.85       5382
147.32.86.187      5102
147.32.84.171      4737
147.32.84.123      4681
147.32.85.95       4382
Name: count, dtype: int64

In [3]:
dns_flows["time_window"] = (
    dns_flows["StartTime"]
    .dt.floor("10s")
)

print(
    "DNS time windows:",
    dns_flows["time_window"].nunique()
)

DNS time windows: 2204


In [4]:
dns_window = (
    dns_flows
    .groupby("time_window")
    .agg(
        dns_query_count=(
            "StartTime",
            "size"
        ),

        dns_total_packets=(
            "TotPkts",
            "sum"
        ),

        dns_total_bytes=(
            "TotBytes",
            "sum"
        ),

        dns_unique_sources=(
            "SrcAddr",
            "nunique"
        ),

        dns_unique_destinations=(
            "DstAddr",
            "nunique"
        )
    )
    .reset_index()
)

dns_window["dns_query_rate"] = (
    dns_window["dns_query_count"]
    / 10.0
)

print(
    "DNS behavioural windows:",
    dns_window.shape
)

display(
    dns_window.head(20)
)

DNS behavioural windows: (2204, 7)


,time_window,dns_query_count,dns_total_packets,dns_total_bytes,dns_unique_sources,dns_unique_destinations,dns_query_rate
0,2011-08-10 09:46:50,285,574,70953,16,2,28.5
1,2011-08-10 09:47:00,428,856,110773,20,2,42.8
2,2011-08-10 09:47:10,347,694,81536,19,2,34.7
3,2011-08-10 09:47:20,287,574,68214,18,1,28.7
4,2011-08-10 09:47:30,396,794,98250,17,1,39.6
5,2011-08-10 09:47:40,255,511,58615,13,1,25.5
6,2011-08-10 09:47:50,326,652,82475,13,2,32.6
7,2011-08-10 09:48:00,339,681,81831,18,1,33.9
8,2011-08-10 09:48:10,243,486,55590,16,1,24.3
9,2011-08-10 09:48:20,261,522,64136,16,2,26.1


In [5]:
# ------------------------------------------------------------
# Destination concentration
# ------------------------------------------------------------

dns_destination_counts = (
    dns_flows
    .groupby(
        [
            "time_window",
            "DstAddr"
        ]
    )
    .size()
    .rename("count")
    .reset_index()
)

dns_max_destination = (
    dns_destination_counts
    .groupby("time_window")["count"]
    .max()
    .rename("dns_max_destination_count")
    .reset_index()
)

dns_window = dns_window.merge(
    dns_max_destination,
    on="time_window",
    how="left"
)

dns_window["dns_packet_concentration"] = (
    dns_window[
        "dns_max_destination_count"
    ]
    /
    dns_window["dns_query_count"]
)

dns_window["dns_packet_concentration"] = (
    dns_window[
        "dns_packet_concentration"
    ].clip(0, 1)
)

In [6]:
dns_destination_bytes = (
    dns_flows
    .groupby(
        [
            "time_window",
            "DstAddr"
        ]
    )["TotBytes"]
    .sum()
    .rename("bytes")
    .reset_index()
)

dns_max_bytes = (
    dns_destination_bytes
    .groupby("time_window")["bytes"]
    .max()
    .rename("dns_max_destination_bytes")
    .reset_index()
)

dns_window = dns_window.merge(
    dns_max_bytes,
    on="time_window",
    how="left"
)

dns_window["dns_byte_concentration"] = (
    dns_window[
        "dns_max_destination_bytes"
    ]
    /
    dns_window[
        "dns_total_bytes"
    ]
).clip(0, 1)

In [7]:
dns_flows = dns_flows.sort_values(
    [
        "SrcAddr",
        "DstAddr",
        "StartTime"
    ]
)

dns_flows["dns_iat"] = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "DstAddr"
        ]
    )["StartTime"]
    .diff()
    .dt.total_seconds()
)

dns_iat = (
    dns_flows
    .groupby("time_window")["dns_iat"]
    .agg(
        dns_iat_mean="mean",
        dns_iat_std="std",
        dns_iat_median="median"
    )
    .reset_index()
)

dns_iat["dns_iat_cv"] = (
    dns_iat["dns_iat_std"]
    /
    dns_iat["dns_iat_mean"]
)

dns_window = dns_window.merge(
    dns_iat,
    on="time_window",
    how="left"
)

In [8]:
dns_window = (
    dns_window
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

dns_feature_columns = [
    "dns_query_rate",
    "dns_unique_destinations",
    "dns_packet_concentration",
    "dns_byte_concentration",
    "dns_iat_cv"
]

dns_model_data = (
    dns_window[
        [
            "time_window"
        ] +
        dns_feature_columns
    ]
    .copy()
)

dns_model_data[
    dns_feature_columns
] = (
    dns_model_data[
        dns_feature_columns
    ].fillna(0)
)

print(
    "DNS feature table:",
    dns_model_data.shape
)

display(
    dns_model_data.head(20)
)

DNS feature table: (2204, 6)


,time_window,dns_query_rate,dns_unique_destinations,dns_packet_concentration,dns_byte_concentration,dns_iat_cv
0,2011-08-10 09:46:50,28.5,2,0.985965,0.982242,2.959128
1,2011-08-10 09:47:00,42.8,2,0.990654,0.989835,4.649555
2,2011-08-10 09:47:10,34.7,2,0.988473,0.984743,4.573901
3,2011-08-10 09:47:20,28.7,1,1.000000,1.000000,4.075933
4,2011-08-10 09:47:30,39.6,1,1.000000,1.000000,5.359755
5,2011-08-10 09:47:40,25.5,1,1.000000,1.000000,4.390550
6,2011-08-10 09:47:50,32.6,2,0.987730,0.983195,6.970080
7,2011-08-10 09:48:00,33.9,1,1.000000,1.000000,5.534991
8,2011-08-10 09:48:10,24.3,1,1.000000,1.000000,4.151409
9,2011-08-10 09:48:20,26.1,2,0.915709,0.872396,4.859535


In [11]:
# ============================================================
# FIX — ADD PROJECT ROOT TO PYTHON PATH
# ============================================================

from pathlib import Path
import sys

current = Path.cwd()

project_root = None

for candidate in [current, *current.parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "Could not find project root containing the 'src' folder."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("src exists:", (project_root / "src").is_dir())

Project root: e:\CODEZILLA-SIH26145
src exists: True


In [12]:
from src.detectors.dns_detector import detect

dns_results = detect(
    dns_model_data[dns_feature_columns],
    top_k=5
)

display(
    pd.DataFrame(dns_results).head(20)
)

,prediction,model_score,decision_threshold,threat_class,severity,supporting_features
0,BENIGN,0.4990,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 28.5, '..."
1,BENIGN,0.4891,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 42.8, '..."
2,BENIGN,0.4887,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 34.7, '..."
3,BENIGN,0.4871,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 28.7, '..."
4,BENIGN,0.4811,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 39.6, '..."
5,BENIGN,0.4853,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 25.5, '..."
6,BENIGN,0.4803,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 32.6, '..."
7,BENIGN,0.4805,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 33.9, '..."
8,BENIGN,0.4866,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 24.3, '..."
9,BENIGN,0.4652,0.5,DNS,LOW,"[{'signal': 'dns_query_rate', 'value': 26.1, '..."


In [13]:
# ============================================================
# STEP 178 — DNS SCORE DISTRIBUTION
# ============================================================

dns_result_df = pd.DataFrame(
    dns_results
)

print("DNS result count:", len(dns_result_df))

print("\nScore distribution:")

print(
    dns_result_df["model_score"].describe()
)

print("\nPrediction distribution:")

print(
    dns_result_df["prediction"].value_counts()
)

DNS result count: 2204

Score distribution:
count    2204.000000
mean        0.484932
std         0.029769
min         0.378200
25%         0.470500
50%         0.474800
75%         0.482000
max         0.611800
Name: model_score, dtype: float64

Prediction distribution:
prediction
BENIGN        1833
SUSPICIOUS     371
Name: count, dtype: int64


In [14]:
# ============================================================
# STEP 179 — TOP DNS SUSPICIOUS WINDOWS
# ============================================================

dns_ranked = (
    dns_model_data
    .copy()
)

dns_ranked["model_score"] = (
    dns_result_df["model_score"]
)

dns_ranked["prediction"] = (
    dns_result_df["prediction"]
)

display(
    dns_ranked
    .sort_values(
        "model_score",
        ascending=False
    )
    .head(30)
)

,time_window,dns_query_rate,dns_unique_destinations,dns_packet_concentration,dns_byte_concentration,dns_iat_cv,model_score,prediction
1357,2011-08-10 13:33:00,56.8,22,0.952465,0.959053,5.503624,0.6118,SUSPICIOUS
1362,2011-08-10 13:33:50,45.0,24,0.922222,0.944027,4.822046,0.6085,SUSPICIOUS
1183,2011-08-10 13:04:00,43.0,21,0.918605,0.934549,4.550244,0.6083,SUSPICIOUS
2171,2011-08-10 15:48:40,43.6,20,0.944954,0.959320,7.111431,0.6062,SUSPICIOUS
1422,2011-08-10 13:43:50,46.4,21,0.948276,0.950394,7.400337,0.6051,SUSPICIOUS
1282,2011-08-10 13:20:30,44.1,23,0.934240,0.917758,5.588678,0.6047,SUSPICIOUS
1266,2011-08-10 13:17:50,51.2,21,0.937500,0.961061,7.434731,0.6045,SUSPICIOUS
1371,2011-08-10 13:35:20,60.5,24,0.943802,0.960976,8.757165,0.6030,SUSPICIOUS
1267,2011-08-10 13:18:00,58.8,20,0.947279,0.974721,11.394287,0.6017,SUSPICIOUS
1372,2011-08-10 13:35:30,39.1,22,0.905371,0.948664,6.561704,0.6005,SUSPICIOUS


In [15]:
# ============================================================
# STEP 180 — HIGH-SCORE DNS BEHAVIOUR
# ============================================================

top_dns = (
    dns_ranked
    .sort_values(
        "model_score",
        ascending=False
    )
    .head(20)
)

display(
    top_dns[
        [
            "time_window",
            "dns_query_rate",
            "dns_unique_destinations",
            "dns_packet_concentration",
            "dns_byte_concentration",
            "dns_iat_cv",
            "model_score",
            "prediction"
        ]
    ]
)

,time_window,dns_query_rate,dns_unique_destinations,dns_packet_concentration,dns_byte_concentration,dns_iat_cv,model_score,prediction
1357,2011-08-10 13:33:00,56.8,22,0.952465,0.959053,5.503624,0.6118,SUSPICIOUS
1362,2011-08-10 13:33:50,45.0,24,0.922222,0.944027,4.822046,0.6085,SUSPICIOUS
1183,2011-08-10 13:04:00,43.0,21,0.918605,0.934549,4.550244,0.6083,SUSPICIOUS
2171,2011-08-10 15:48:40,43.6,20,0.944954,0.959320,7.111431,0.6062,SUSPICIOUS
1422,2011-08-10 13:43:50,46.4,21,0.948276,0.950394,7.400337,0.6051,SUSPICIOUS
1282,2011-08-10 13:20:30,44.1,23,0.934240,0.917758,5.588678,0.6047,SUSPICIOUS
1266,2011-08-10 13:17:50,51.2,21,0.937500,0.961061,7.434731,0.6045,SUSPICIOUS
1371,2011-08-10 13:35:20,60.5,24,0.943802,0.960976,8.757165,0.6030,SUSPICIOUS
1267,2011-08-10 13:18:00,58.8,20,0.947279,0.974721,11.394287,0.6017,SUSPICIOUS
1372,2011-08-10 13:35:30,39.1,22,0.905371,0.948664,6.561704,0.6005,SUSPICIOUS


In [16]:
# ============================================================
# STEP 181 — DNS EXPLANATION
# ============================================================

top_index = (
    dns_ranked[
        "model_score"
    ]
    .idxmax()
)

print(
    "Highest scoring DNS window:"
)

print(
    dns_ranked.loc[
        top_index
    ]
)

print("\nDetector evidence:")

print(
    dns_results[top_index][
        "supporting_features"
    ]
)

Highest scoring DNS window:
time_window                 2011-08-10 13:33:00
dns_query_rate                             56.8
dns_unique_destinations                      22
dns_packet_concentration               0.952465
dns_byte_concentration                 0.959053
dns_iat_cv                             5.503624
model_score                              0.6118
prediction                           SUSPICIOUS
Name: 1357, dtype: object

Detector evidence:
[{'signal': 'dns_query_rate', 'value': 56.8, 'contribution': 0.2}, {'signal': 'dns_destination_diversity', 'value': 22.0, 'contribution': 0.15}, {'signal': 'dns_packet_concentration', 'value': 0.9524647887323944, 'contribution': 0.14286971830985915}, {'signal': 'dns_byte_concentration', 'value': 0.9590531328788869, 'contribution': 0.0959053132878887}, {'signal': 'dns_timing_pattern', 'value': 5.503624202401414, 'contribution': 0.023064063256393815}]


In [17]:
# ============================================================
# STEP 182 — SOURCE-CENTRIC DNS WINDOWS
# ============================================================

dns_flows = dns_flows.copy()

dns_flows["time_window"] = (
    dns_flows["StartTime"]
    .dt.floor("10s")
)

dns_source_window = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )
    .agg(
        dns_query_count=(
            "StartTime",
            "size"
        ),

        dns_total_packets=(
            "TotPkts",
            "sum"
        ),

        dns_total_bytes=(
            "TotBytes",
            "sum"
        ),

        dns_unique_destinations=(
            "DstAddr",
            "nunique"
        )
    )
    .reset_index()
)

dns_source_window["dns_query_rate"] = (
    dns_source_window["dns_query_count"] / 10.0
)

print(
    "Source-centric DNS windows:",
    dns_source_window.shape
)

display(
    dns_source_window.head(20)
)

Source-centric DNS windows: (40953, 7)


,SrcAddr,time_window,dns_query_count,dns_total_packets,dns_total_bytes,dns_unique_destinations,dns_query_rate
0,147.32.84.10,2011-08-10 12:40:10,4,8,1065,1,0.4
1,147.32.84.10,2011-08-10 13:40:00,4,8,1065,1,0.4
2,147.32.84.10,2011-08-10 13:47:50,36,72,7932,1,3.6
3,147.32.84.10,2011-08-10 13:48:00,3,6,661,1,0.3
4,147.32.84.10,2011-08-10 13:48:10,21,42,4627,1,2.1
5,147.32.84.10,2011-08-10 13:48:20,27,54,5949,1,2.7
6,147.32.84.10,2011-08-10 13:48:30,30,60,6610,1,3.0
7,147.32.84.10,2011-08-10 13:48:40,21,42,4627,1,2.1
8,147.32.84.10,2011-08-10 13:48:50,15,30,3305,1,1.5
9,147.32.84.10,2011-08-10 13:49:00,12,24,2644,1,1.2


In [18]:
# ============================================================
# STEP 183 — SOURCE DNS CONCENTRATION
# ============================================================

dns_dest_counts = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window",
            "DstAddr"
        ]
    )
    .size()
    .rename("count")
    .reset_index()
)

dns_max_dest = (
    dns_dest_counts
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )["count"]
    .max()
    .rename("dns_max_destination_count")
    .reset_index()
)

dns_source_window = dns_source_window.merge(
    dns_max_dest,
    on=["SrcAddr", "time_window"],
    how="left"
)

dns_source_window["dns_packet_concentration"] = (
    dns_source_window["dns_max_destination_count"]
    /
    dns_source_window["dns_query_count"]
)

dns_source_window[
    "dns_packet_concentration"
] = (
    dns_source_window[
        "dns_packet_concentration"
    ].clip(0, 1)
)

In [19]:
# ============================================================
# STEP 184 — SOURCE DNS BYTE CONCENTRATION
# ============================================================

dns_dest_bytes = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window",
            "DstAddr"
        ]
    )["TotBytes"]
    .sum()
    .rename("bytes")
    .reset_index()
)

dns_max_bytes = (
    dns_dest_bytes
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )["bytes"]
    .max()
    .rename("dns_max_destination_bytes")
    .reset_index()
)

dns_source_window = dns_source_window.merge(
    dns_max_bytes,
    on=["SrcAddr", "time_window"],
    how="left"
)

dns_source_window["dns_byte_concentration"] = (
    dns_source_window["dns_max_destination_bytes"]
    /
    dns_source_window["dns_total_bytes"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0).clip(0, 1)

In [20]:
# ============================================================
# STEP 185 — SOURCE DNS TIMING
# ============================================================

dns_flows = dns_flows.sort_values(
    [
        "SrcAddr",
        "DstAddr",
        "StartTime"
    ]
)

dns_flows["dns_iat"] = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "DstAddr"
        ]
    )["StartTime"]
    .diff()
    .dt.total_seconds()
)

dns_iat = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )["dns_iat"]
    .agg(
        dns_iat_mean="mean",
        dns_iat_std="std",
        dns_iat_median="median"
    )
    .reset_index()
)

dns_iat["dns_iat_cv"] = (
    dns_iat["dns_iat_std"]
    /
    dns_iat["dns_iat_mean"]
)

dns_source_window = dns_source_window.merge(
    dns_iat,
    on=["SrcAddr", "time_window"],
    how="left"
)

dns_source_window = (
    dns_source_window
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

In [21]:
# ============================================================
# STEP 186 — DNS ANOMALY SCORE
# ============================================================

def dns_anomaly_score(df):
    data = df.copy()

    score = pd.Series(
        0.0,
        index=data.index
    )

    # High query rate
    query_rate_score = np.clip(
        data["dns_query_rate"] / 100.0,
        0,
        1
    )

    score += 0.30 * query_rate_score

    # Many destinations
    destination_score = np.clip(
        data["dns_unique_destinations"] / 50.0,
        0,
        1
    )

    score += 0.20 * destination_score

    # Byte concentration
    score += (
        0.15 *
        data["dns_byte_concentration"].fillna(0)
    )

    # Packet concentration
    score += (
        0.15 *
        data["dns_packet_concentration"].fillna(0)
    )

    # Timing irregularity
    timing_score = (
        data["dns_iat_cv"]
        .fillna(999)
        /
        (
            1 +
            data["dns_iat_cv"].fillna(999)
        )
    )

    score += 0.20 * np.clip(
        timing_score,
        0,
        1
    )

    return score.clip(0, 1)

In [22]:
# ============================================================
# STEP 187 — CALCULATE SOURCE-CENTRIC DNS SCORES
# ============================================================

dns_source_window[
    "dns_anomaly_score"
] = dns_anomaly_score(
    dns_source_window
)

print(
    dns_source_window[
        "dns_anomaly_score"
    ].describe()
)

display(
    dns_source_window
    .sort_values(
        "dns_anomaly_score",
        ascending=False
    )
    .head(30)
)

count    40953.000000
mean         0.437901
std          0.055171
min          0.144610
25%          0.421749
50%          0.436789
75%          0.468690
max          0.783127
Name: dns_anomaly_score, dtype: float64


,SrcAddr,time_window,dns_query_count,dns_total_packets,dns_total_bytes,dns_unique_destinations,dns_query_rate,dns_max_destination_count,dns_packet_concentration,dns_max_destination_bytes,dns_byte_concentration,dns_iat_mean,dns_iat_std,dns_iat_median,dns_iat_cv,dns_anomaly_score
1741,147.32.84.138,2011-08-10 10:23:40,2237,4474,567220,1,223.7,2237,1.000000,567220,1.00000,0.004473,0.038384,0.000458,8.581845,0.783127
1733,147.32.84.138,2011-08-10 10:22:20,1896,3792,480432,1,189.6,1896,1.000000,480432,1.00000,0.005383,0.042830,0.000468,7.956288,0.781669
1740,147.32.84.138,2011-08-10 10:23:30,1908,3816,483092,1,190.8,1908,1.000000,483092,1.00000,0.005409,0.042963,0.000458,7.943300,0.781637
1736,147.32.84.138,2011-08-10 10:22:50,2072,4144,524135,1,207.2,2072,1.000000,524135,1.00000,0.004822,0.037946,0.000455,7.868654,0.781449
1742,147.32.84.138,2011-08-10 10:23:50,1944,3890,490474,1,194.4,1944,1.000000,490474,1.00000,0.005119,0.039674,0.000459,7.749739,0.781142
1737,147.32.84.138,2011-08-10 10:23:00,1534,3068,384396,1,153.4,1534,1.000000,384396,1.00000,0.006510,0.047271,0.000457,7.261088,0.779790
1734,147.32.84.138,2011-08-10 10:22:30,1496,2992,376264,1,149.6,1496,1.000000,376264,1.00000,0.006510,0.046016,0.000504,7.068366,0.779212
1735,147.32.84.138,2011-08-10 10:22:40,1802,3604,454955,1,180.2,1802,1.000000,454955,1.00000,0.005698,0.040201,0.000485,7.055813,0.779173
1739,147.32.84.138,2011-08-10 10:23:20,1522,3044,381736,1,152.2,1522,1.000000,381736,1.00000,0.006577,0.046225,0.000457,7.028642,0.779089
1743,147.32.84.138,2011-08-10 10:24:00,1596,3192,399953,1,159.6,1596,1.000000,399953,1.00000,0.006214,0.042638,0.000503,6.861984,0.778561


In [23]:
# ============================================================
# STEP 188 — DNS GROUND TRUTH
# ============================================================

dns_flows["dns_threat"] = (
    dns_flows["Label"]
    .astype(str)
    .str.contains(
        r"From-Botnet.*UDP-DNS",
        case=False,
        regex=True
    )
    .astype(int)
)

dns_target = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )["dns_threat"]
    .max()
    .rename("dns_target")
    .reset_index()
)

dns_source_window = dns_source_window.merge(
    dns_target,
    on=[
        "SrcAddr",
        "time_window"
    ],
    how="left"
)

dns_source_window["dns_target"] = (
    dns_source_window["dns_target"]
    .fillna(0)
    .astype(int)
)

print(
    "DNS target distribution:"
)

print(
    dns_source_window[
        "dns_target"
    ].value_counts()
)

DNS target distribution:
dns_target
0    40107
1      846
Name: count, dtype: int64


In [24]:
# ============================================================
# STEP 189 — DNS SCORE SEPARATION
# ============================================================

display(
    dns_source_window
    .groupby("dns_target")[
        [
            "dns_anomaly_score",
            "dns_query_rate",
            "dns_unique_destinations",
            "dns_packet_concentration",
            "dns_byte_concentration",
            "dns_iat_cv"
        ]
    ]
    .describe()
)

dns_anomaly_score                                                  \
                       count      mean       std      min      25%       50%   
dns_target                                                                     
0                    40107.0  0.439032  0.053409  0.15860  0.42175  0.437140   
1                      846.0  0.384254  0.095860  0.14461  0.31776  0.397019   

                               dns_query_rate            ...  \
                 75%       max          count      mean  ...   
dns_target                                               ...   
0           0.468675  0.783127        40107.0  2.402232  ...   
1           0.470948  0.671775          846.0  3.450000  ...   

           dns_byte_concentration      dns_iat_cv                      \
                              75%  max      count      mean       std   
dns_target                                                              
0                             1.0  1.0    35516.0  2.047766  1.471619   
1                             1.0  1.0      669.0  1.806782  1.332211   

                                                               
                 min       25%       50%       75%        max  
dns_target                                                     
0           0.000000  1.413879  1.730001  2.713900  15.001481  
1           0.000016  0.942523  1.378820  2.148395   7.714774  

[2 rows x 48 columns]

In [25]:
# ============================================================
# STEP 190 — DNS DETECTOR EVALUATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

dns_eval = []

for threshold in np.arange(
    0.10,
    0.91,
    0.05
):

    pred = (
        dns_source_window[
            "dns_anomaly_score"
        ] >= threshold
    ).astype(int)

    dns_eval.append({
        "threshold": round(
            float(threshold),
            2
        ),
        "precision": precision_score(
            dns_source_window["dns_target"],
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            dns_source_window["dns_target"],
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            dns_source_window["dns_target"],
            pred,
            zero_division=0
        )
    })

dns_eval = pd.DataFrame(
    dns_eval
)

display(
    dns_eval
    .sort_values(
        "f1",
        ascending=False
    )
)

,threshold,precision,recall,f1
8,0.50,0.024649,0.222222,0.044376
0,0.10,0.020658,1.000000,0.040479
1,0.15,0.020634,0.998818,0.040433
2,0.20,0.019954,0.964539,0.039098
3,0.25,0.018590,0.897163,0.036425
4,0.30,0.016483,0.793144,0.032295
7,0.45,0.015507,0.276596,0.029367
5,0.35,0.014126,0.634752,0.027638
6,0.40,0.011832,0.492908,0.023110
9,0.55,0.020619,0.002364,0.004242


In [26]:
# ============================================================
# STEP 191 — RICHER SOURCE-CENTRIC DNS FEATURES
# ============================================================

dns_work = dns_source_window.copy()

dns_work = dns_work.sort_values(
    ["SrcAddr", "time_window"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Rolling source behaviour
# ------------------------------------------------------------

group = dns_work.groupby("SrcAddr", group_keys=False)

dns_work["query_rate_prev"] = (
    group["dns_query_rate"]
    .shift(1)
)

dns_work["query_rate_roll3"] = (
    group["dns_query_rate"]
    .rolling(3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

dns_work["query_rate_roll6"] = (
    group["dns_query_rate"]
    .rolling(6, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

dns_work["query_rate_std6"] = (
    group["dns_query_rate"]
    .rolling(6, min_periods=2)
    .std()
    .reset_index(level=0, drop=True)
)

dns_work["query_rate_change"] = (
    dns_work["dns_query_rate"]
    -
    dns_work["query_rate_prev"]
).fillna(0)

# ------------------------------------------------------------
# Relative deviation from source history
# ------------------------------------------------------------

dns_work["query_rate_z6"] = (
    (
        dns_work["dns_query_rate"]
        -
        dns_work["query_rate_roll6"]
    )
    /
    (
        dns_work["query_rate_std6"]
        .replace(0, np.nan)
    )
)

dns_work["query_rate_z6"] = (
    dns_work["query_rate_z6"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
    .clip(-10, 10)
)

# ------------------------------------------------------------
# Destination behaviour changes
# ------------------------------------------------------------

dns_work["destination_change"] = (
    dns_work["dns_unique_destinations"]
    -
    group["dns_unique_destinations"]
    .shift(1)
).fillna(0)

# ------------------------------------------------------------
# Byte / packet behaviour
# ------------------------------------------------------------

dns_work["bytes_per_query"] = (
    dns_work["dns_total_bytes"]
    /
    dns_work["dns_query_count"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

dns_work["packets_per_query"] = (
    dns_work["dns_total_packets"]
    /
    dns_work["dns_query_count"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

# ------------------------------------------------------------
# Final feature list
# ------------------------------------------------------------

dns_ml_features = [
    "dns_query_rate",
    "dns_unique_destinations",
    "dns_packet_concentration",
    "dns_byte_concentration",
    "dns_iat_cv",
    "query_rate_prev",
    "query_rate_roll3",
    "query_rate_roll6",
    "query_rate_std6",
    "query_rate_change",
    "query_rate_z6",
    "destination_change",
    "bytes_per_query",
    "packets_per_query"
]

dns_work[dns_ml_features] = (
    dns_work[dns_ml_features]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

print("DNS ML dataset:", dns_work.shape)

print("\nFeatures:")

for col in dns_ml_features:
    print("-", col)

DNS ML dataset: (40953, 26)

Features:
- dns_query_rate
- dns_unique_destinations
- dns_packet_concentration
- dns_byte_concentration
- dns_iat_cv
- query_rate_prev
- query_rate_roll3
- query_rate_roll6
- query_rate_std6
- query_rate_change
- query_rate_z6
- destination_change
- bytes_per_query
- packets_per_query


In [27]:
# ============================================================
# STEP 192 — DNS CLASS SEPARATION
# ============================================================

display(
    dns_work
    .groupby("dns_target")[dns_ml_features]
    .mean()
    .T
)

dns_target,0,1
dns_query_rate,2.402232,3.450000
dns_unique_destinations,1.014461,13.210402
dns_packet_concentration,0.997187,0.667903
dns_byte_concentration,0.997487,0.684202
dns_iat_cv,1.813360,1.428768
query_rate_prev,2.398571,3.445390
query_rate_roll3,2.402707,3.435481
query_rate_roll6,2.403340,3.402120
query_rate_std6,1.079597,4.450579
query_rate_change,-0.000770,0.004374


In [28]:
# ============================================================
# STEP 193 — DNS TEMPORAL SPLIT
# ============================================================

dns_work = (
    dns_work
    .sort_values("time_window")
    .reset_index(drop=True)
)

n = len(dns_work)

train_end = int(n * 0.60)
val_end = int(n * 0.80)

dns_train = dns_work.iloc[:train_end].copy()
dns_val = dns_work.iloc[train_end:val_end].copy()
dns_test = dns_work.iloc[val_end:].copy()

X_dns_train = dns_train[dns_ml_features].copy()
y_dns_train = dns_train["dns_target"].copy()

X_dns_val = dns_val[dns_ml_features].copy()
y_dns_val = dns_val["dns_target"].copy()

X_dns_test = dns_test[dns_ml_features].copy()
y_dns_test = dns_test["dns_target"].copy()

print("Train:", X_dns_train.shape)
print("Validation:", X_dns_val.shape)
print("Test:", X_dns_test.shape)

print("\nTrain target:")
print(y_dns_train.value_counts())

print("\nValidation target:")
print(y_dns_val.value_counts())

print("\nTest target:")
print(y_dns_test.value_counts())

Train: (24571, 14)
Validation: (8191, 14)
Test: (8191, 14)

Train target:
dns_target
0    24073
1      498
Name: count, dtype: int64

Validation target:
dns_target
0    8056
1     135
Name: count, dtype: int64

Test target:
dns_target
0    7978
1     213
Name: count, dtype: int64


In [29]:
# ============================================================
# STEP 194 — SUPERVISED DNS MODEL
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

dns_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_dns_train
)

dns_hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.04,
    max_leaf_nodes=15,
    min_samples_leaf=10,
    l2_regularization=2.0,
    random_state=42
)

dns_hgb.fit(
    X_dns_train,
    y_dns_train,
    sample_weight=dns_weights
)

print("✅ DNS HGB trained")

c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

✅ DNS HGB trained


In [30]:
# ============================================================
# STEP 195 — DNS VALIDATION
# ============================================================

dns_val_proba = (
    dns_hgb
    .predict_proba(X_dns_val)[:, 1]
)

print(
    "Validation probability distribution:"
)

print(
    pd.Series(dns_val_proba).describe()
)

Validation probability distribution:
count    8191.000000
mean        0.082878
std         0.187362
min         0.001757
25%         0.004546
50%         0.011278
75%         0.032880
max         0.996823
dtype: float64


In [31]:
# ============================================================
# STEP 196 — DNS THRESHOLD SEARCH
# ============================================================

dns_threshold_rows = []

for threshold in np.arange(
    0.05,
    0.96,
    0.02
):

    pred = (
        dns_val_proba >= threshold
    ).astype(int)

    dns_threshold_rows.append({
        "threshold": round(
            float(threshold),
            2
        ),

        "precision": precision_score(
            y_dns_val,
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_dns_val,
            pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_dns_val,
            pred,
            zero_division=0
        )
    })

dns_threshold_table = pd.DataFrame(
    dns_threshold_rows
)

display(
    dns_threshold_table
    .sort_values(
        "f1",
        ascending=False
    )
    .head(15)
)

,threshold,precision,recall,f1
44,0.93,0.967033,0.651852,0.778761
42,0.89,0.910000,0.674074,0.774468
43,0.91,0.936842,0.659259,0.773913
45,0.95,0.977273,0.637037,0.771300
41,0.87,0.798319,0.703704,0.748031
40,0.85,0.772358,0.703704,0.736434
39,0.83,0.721805,0.711111,0.716418
38,0.81,0.697842,0.718519,0.708029
37,0.79,0.653333,0.725926,0.687719
36,0.77,0.614907,0.733333,0.668919


In [32]:
# ============================================================
# STEP 197 — SELECT DNS THRESHOLD
# ============================================================

eligible_dns = dns_threshold_table[
    dns_threshold_table["recall"] >= 0.60
]

if len(eligible_dns) > 0:

    best_dns = (
        eligible_dns
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

else:

    best_dns = (
        dns_threshold_table
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

DNS_THRESHOLD = float(
    best_dns["threshold"]
)

print(
    "Selected DNS threshold:",
    DNS_THRESHOLD
)

print(
    "Validation precision:",
    best_dns["precision"]
)

print(
    "Validation recall:",
    best_dns["recall"]
)

print(
    "Validation F1:",
    best_dns["f1"]
)

Selected DNS threshold: 0.93
Validation precision: 0.967032967032967
Validation recall: 0.6518518518518519
Validation F1: 0.7787610619469026


In [35]:
# ============================================================
# FIX — DNS EVALUATION IMPORTS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from sklearn.inspection import permutation_importance

print("✅ DNS evaluation metrics imported")

✅ DNS evaluation metrics imported


In [36]:
# ============================================================
# STEP 198 — FINAL DNS TEST
# ============================================================

dns_test_proba = (
    dns_hgb
    .predict_proba(X_dns_test)[:, 1]
)

dns_test_pred = (
    dns_test_proba >= DNS_THRESHOLD
).astype(int)

dns_test_accuracy = accuracy_score(
    y_dns_test,
    dns_test_pred
)

dns_test_precision = precision_score(
    y_dns_test,
    dns_test_pred,
    zero_division=0
)

dns_test_recall = recall_score(
    y_dns_test,
    dns_test_pred,
    zero_division=0
)

dns_test_f1 = f1_score(
    y_dns_test,
    dns_test_pred,
    zero_division=0
)

print("==========================================")
print("FINAL DNS HGB TEST")
print("==========================================")

print(
    "Threshold:",
    DNS_THRESHOLD
)

print(
    "Accuracy:",
    f"{dns_test_accuracy:.4f}"
)

print(
    "Precision:",
    f"{dns_test_precision:.4f}"
)

print(
    "Recall:",
    f"{dns_test_recall:.4f}"
)

print(
    "F1:",
    f"{dns_test_f1:.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_dns_test,
        dns_test_pred
    )
)

FINAL DNS HGB TEST
Threshold: 0.93
Accuracy: 0.9916
Precision: 0.9444
Recall: 0.7183
F1: 0.8160

Confusion Matrix:
[[7969    9]
 [  60  153]]


In [37]:
# ============================================================
# STEP 199 — DNS PERMUTATION IMPORTANCE
# ============================================================

dns_perm = permutation_importance(
    dns_hgb,
    X_dns_val,
    y_dns_val,
    scoring="f1",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

dns_importance = pd.DataFrame({
    "Feature": dns_ml_features,
    "Importance": dns_perm.importances_mean,
    "Std": dns_perm.importances_std
})

dns_importance = (
    dns_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    dns_importance
)

,Feature,Importance,Std
0,bytes_per_query,0.094653,0.008014
1,dns_unique_destinations,0.080958,0.002279
2,query_rate_std6,0.052070,0.005656
3,query_rate_z6,0.010561,0.007178
4,destination_change,0.005502,0.002151
5,dns_packet_concentration,0.003628,0.002130
6,packets_per_query,0.000190,0.000290
7,dns_byte_concentration,-0.000254,0.000311
8,query_rate_roll3,-0.010889,0.003811
9,query_rate_change,-0.014261,0.004164


In [38]:
# ============================================================
# STEP 200 — SAVE VALIDATED DNS MODEL
# ============================================================

import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

dns_model_bundle = {
    "model": dns_hgb,
    "threshold": DNS_THRESHOLD,
    "features": dns_ml_features,
}

model_path = MODEL_DIR / "dns_hgb.joblib"

joblib.dump(
    dns_model_bundle,
    model_path
)

print("✅ DNS model saved")
print("Path:", model_path)
print("Threshold:", DNS_THRESHOLD)
print("Features:", len(dns_ml_features))

✅ DNS model saved
Path: ..\models\dns_hgb.joblib
Threshold: 0.93
Features: 14


In [39]:
# ============================================================
# STEP 201 — VERIFY DNS MODEL ARTIFACT
# ============================================================

loaded_dns_bundle = joblib.load(
    model_path
)

print(
    "Loaded threshold:",
    loaded_dns_bundle["threshold"]
)

print(
    "Loaded feature count:",
    len(
        loaded_dns_bundle["features"]
    )
)

print(
    "Model type:",
    type(
        loaded_dns_bundle["model"]
    ).__name__
)

Loaded threshold: 0.93
Loaded feature count: 14
Model type: HistGradientBoostingClassifier


In [1]:
from pathlib import Path
import sys

current = Path.cwd()

project_root = None

for candidate in [current, *current.parents]:

    if (candidate / "src").is_dir():

        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "Could not find project root containing src/"
    )

if str(project_root) not in sys.path:
    sys.path.insert(
        0,
        str(project_root)
    )

print("Project root:", project_root)
print(
    "Model exists:",
    (project_root / "models" / "dns_hgb.joblib").exists()
)

Project root: e:\CODEZILLA-SIH26145
Model exists: True


In [2]:
from src.detectors.dns_detector import (
    detect,
    THRESHOLD,
    FEATURES
)

print("DNS threshold:", THRESHOLD)
print("DNS feature count:", len(FEATURES))

print("\nDNS features:")

for feature in FEATURES:
    print("-", feature)

DNS threshold: 0.93
DNS feature count: 14

DNS features:
- dns_query_rate
- dns_unique_destinations
- dns_packet_concentration
- dns_byte_concentration
- dns_iat_cv
- query_rate_prev
- query_rate_roll3
- query_rate_roll6
- query_rate_std6
- query_rate_change
- query_rate_z6
- destination_change
- bytes_per_query
- packets_per_query


In [5]:
print("dns_df:", "dns_df" in globals())
print("dns_flows:", "dns_flows" in globals())
print("dns_source_window:", "dns_source_window" in globals())
print("dns_work:", "dns_work" in globals())
print("dns_hgb:", "dns_hgb" in globals())
print("dns_ml_features:", "dns_ml_features" in globals())

dns_df: False
dns_flows: False
dns_source_window: False
dns_work: False
dns_hgb: False
dns_ml_features: False


In [6]:
# ============================================================
# RECOVERY — REBUILD DNS PIPELINE FROM SCRATCH
# ============================================================

from pathlib import Path
import sys
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Find project root
# ------------------------------------------------------------

current = Path.cwd()

project_root = None

for candidate in [current, *current.parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "Could not find project root containing src/"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

# ------------------------------------------------------------
# 2. Locate Scenario 1 NetFlow
# ------------------------------------------------------------

DNS_FILE = (
    project_root
    / "data"
    / "raw"
    / "CTU-13"
    / "capture20110810.binetflow"
)

if not DNS_FILE.exists():
    raise FileNotFoundError(
        f"Scenario 1 file not found:\n{DNS_FILE}"
    )

print("DNS file:", DNS_FILE)
print("File exists:", DNS_FILE.exists())

# ------------------------------------------------------------
# 3. Load required columns
# ------------------------------------------------------------

columns = [
    "StartTime",
    "Dur",
    "Proto",
    "SrcAddr",
    "Sport",
    "Dir",
    "DstAddr",
    "Dport",
    "TotPkts",
    "TotBytes",
    "SrcBytes",
    "Label"
]

dns_df = pd.read_csv(
    DNS_FILE,
    usecols=columns
)

print("DNS dataframe:", dns_df.shape)

# ------------------------------------------------------------
# 4. Basic cleaning
# ------------------------------------------------------------

dns_df["StartTime"] = pd.to_datetime(
    dns_df["StartTime"],
    errors="coerce"
)

dns_df["Dport_numeric"] = pd.to_numeric(
    dns_df["Dport"].astype(str).str.strip(),
    errors="coerce"
)

dns_df = dns_df.dropna(
    subset=[
        "StartTime",
        "SrcAddr",
        "DstAddr"
    ]
)

# ------------------------------------------------------------
# 5. Keep DNS destination port 53
# ------------------------------------------------------------

dns_flows = dns_df[
    dns_df["Dport_numeric"] == 53
].copy()

dns_flows["time_window"] = (
    dns_flows["StartTime"]
    .dt.floor("10s")
)

print("DNS flows:", len(dns_flows))
print(
    "DNS windows:",
    dns_flows["time_window"].nunique()
)

# ------------------------------------------------------------
# 6. Source-centric window aggregation
# ------------------------------------------------------------

dns_source_window = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )
    .agg(
        dns_query_count=(
            "StartTime",
            "size"
        ),

        dns_total_packets=(
            "TotPkts",
            "sum"
        ),

        dns_total_bytes=(
            "TotBytes",
            "sum"
        ),

        dns_unique_destinations=(
            "DstAddr",
            "nunique"
        )
    )
    .reset_index()
)

dns_source_window["dns_query_rate"] = (
    dns_source_window["dns_query_count"]
    / 10.0
)

# ------------------------------------------------------------
# 7. Destination concentration
# ------------------------------------------------------------

dns_dest_counts = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window",
            "DstAddr"
        ]
    )
    .size()
    .rename("count")
    .reset_index()
)

dns_max_dest = (
    dns_dest_counts
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )["count"]
    .max()
    .rename("dns_max_destination_count")
    .reset_index()
)

dns_source_window = dns_source_window.merge(
    dns_max_dest,
    on=[
        "SrcAddr",
        "time_window"
    ],
    how="left"
)

dns_source_window["dns_packet_concentration"] = (
    dns_source_window[
        "dns_max_destination_count"
    ]
    /
    dns_source_window[
        "dns_query_count"
    ]
).clip(0, 1)

# ------------------------------------------------------------
# 8. Byte concentration
# ------------------------------------------------------------

dns_dest_bytes = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window",
            "DstAddr"
        ]
    )["TotBytes"]
    .sum()
    .rename("bytes")
    .reset_index()
)

dns_max_bytes = (
    dns_dest_bytes
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )["bytes"]
    .max()
    .rename("dns_max_destination_bytes")
    .reset_index()
)

dns_source_window = dns_source_window.merge(
    dns_max_bytes,
    on=[
        "SrcAddr",
        "time_window"
    ],
    how="left"
)

dns_source_window["dns_byte_concentration"] = (
    dns_source_window[
        "dns_max_destination_bytes"
    ]
    /
    dns_source_window[
        "dns_total_bytes"
    ]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0).clip(0, 1)

# ------------------------------------------------------------
# 9. DNS inter-arrival timing
# ------------------------------------------------------------

dns_flows = dns_flows.sort_values(
    [
        "SrcAddr",
        "DstAddr",
        "StartTime"
    ]
)

dns_flows["dns_iat"] = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "DstAddr"
        ]
    )["StartTime"]
    .diff()
    .dt.total_seconds()
)

dns_iat = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )["dns_iat"]
    .agg(
        dns_iat_mean="mean",
        dns_iat_std="std",
        dns_iat_median="median"
    )
    .reset_index()
)

dns_iat["dns_iat_cv"] = (
    dns_iat["dns_iat_std"]
    /
    dns_iat["dns_iat_mean"]
)

dns_source_window = dns_source_window.merge(
    dns_iat,
    on=[
        "SrcAddr",
        "time_window"
    ],
    how="left"
)

# ------------------------------------------------------------
# 10. DNS ground truth
# ------------------------------------------------------------

dns_flows["dns_target"] = (
    dns_flows["Label"]
    .astype(str)
    .str.contains(
        r"From-Botnet.*UDP-DNS",
        case=False,
        regex=True
    )
    .astype(int)
)

dns_target = (
    dns_flows
    .groupby(
        [
            "SrcAddr",
            "time_window"
        ]
    )["dns_target"]
    .max()
    .rename("dns_target")
    .reset_index()
)

dns_source_window = dns_source_window.merge(
    dns_target,
    on=[
        "SrcAddr",
        "time_window"
    ],
    how="left"
)

dns_source_window["dns_target"] = (
    dns_source_window["dns_target"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 11. Sort
# ------------------------------------------------------------

dns_source_window = (
    dns_source_window
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .sort_values(
        [
            "SrcAddr",
            "time_window"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 12. Build 14 features expected by saved model
# ------------------------------------------------------------

dns_work = dns_source_window.copy()

group = dns_work.groupby(
    "SrcAddr",
    group_keys=False
)

dns_work["query_rate_prev"] = (
    group["dns_query_rate"].shift(1)
)

dns_work["query_rate_roll3"] = (
    group["dns_query_rate"]
    .rolling(3, min_periods=1)
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

dns_work["query_rate_roll6"] = (
    group["dns_query_rate"]
    .rolling(6, min_periods=1)
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

dns_work["query_rate_std6"] = (
    group["dns_query_rate"]
    .rolling(6, min_periods=2)
    .std()
    .reset_index(
        level=0,
        drop=True
    )
)

dns_work["query_rate_change"] = (
    dns_work["dns_query_rate"]
    -
    dns_work["query_rate_prev"]
).fillna(0)

dns_work["query_rate_z6"] = (
    (
        dns_work["dns_query_rate"]
        -
        dns_work["query_rate_roll6"]
    )
    /
    dns_work["query_rate_std6"].replace(
        0,
        np.nan
    )
)

dns_work["query_rate_z6"] = (
    dns_work["query_rate_z6"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
    .clip(-10, 10)
)

dns_work["destination_change"] = (
    dns_work["dns_unique_destinations"]
    -
    group["dns_unique_destinations"].shift(1)
).fillna(0)

dns_work["bytes_per_query"] = (
    dns_work["dns_total_bytes"]
    /
    dns_work["dns_query_count"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

dns_work["packets_per_query"] = (
    dns_work["dns_total_packets"]
    /
    dns_work["dns_query_count"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

dns_ml_features = [
    "dns_query_rate",
    "dns_unique_destinations",
    "dns_packet_concentration",
    "dns_byte_concentration",
    "dns_iat_cv",
    "query_rate_prev",
    "query_rate_roll3",
    "query_rate_roll6",
    "query_rate_std6",
    "query_rate_change",
    "query_rate_z6",
    "destination_change",
    "bytes_per_query",
    "packets_per_query"
]

dns_work[dns_ml_features] = (
    dns_work[dns_ml_features]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

print("\n✅ DNS pipeline rebuilt")
print("dns_df:", dns_df.shape)
print("dns_flows:", dns_flows.shape)
print("dns_source_window:", dns_source_window.shape)
print("dns_work:", dns_work.shape)
print("Feature count:", len(dns_ml_features))

print("\nDNS target distribution:")
print(
    dns_work["dns_target"].value_counts()
)

Project root: e:\CODEZILLA-SIH26145
DNS file: e:\CODEZILLA-SIH26145\data\raw\CTU-13\capture20110810.binetflow
File exists: True
DNS dataframe: (2824636, 12)
DNS flows: 992650
DNS windows: 2204

✅ DNS pipeline rebuilt
dns_df: (2824636, 13)
dns_flows: (992650, 16)
dns_source_window: (40953, 16)
dns_work: (40953, 25)
Feature count: 14

DNS target distribution:
dns_target
0    40107
1      846
Name: count, dtype: int64


In [7]:
# ============================================================
# LOAD SAVED DNS MODEL
# ============================================================

import joblib

MODEL_PATH = (
    project_root
    / "models"
    / "dns_hgb.joblib"
)

print("Model path:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

dns_bundle = joblib.load(
    MODEL_PATH
)

dns_hgb = dns_bundle["model"]
DNS_THRESHOLD = float(
    dns_bundle["threshold"]
)
DNS_FEATURES = list(
    dns_bundle["features"]
)

print("✅ DNS model loaded")
print("Threshold:", DNS_THRESHOLD)
print("Feature count:", len(DNS_FEATURES))

print("\nSaved features:")
for feature in DNS_FEATURES:
    print("-", feature)

Model path: e:\CODEZILLA-SIH26145\models\dns_hgb.joblib
Model exists: True
✅ DNS model loaded
Threshold: 0.93
Feature count: 14

Saved features:
- dns_query_rate
- dns_unique_destinations
- dns_packet_concentration
- dns_byte_concentration
- dns_iat_cv
- query_rate_prev
- query_rate_roll3
- query_rate_roll6
- query_rate_std6
- query_rate_change
- query_rate_z6
- destination_change
- bytes_per_query
- packets_per_query


In [8]:
# ============================================================
# TEST PRODUCTION DNS DETECTOR
# ============================================================

from src.detectors.dns_detector import detect

dns_production_results = detect(
    dns_work[DNS_FEATURES],
    top_k=5
)

dns_production_df = pd.DataFrame(
    dns_production_results
)

print(
    "Production DNS results:",
    len(dns_production_results)
)

print("\nPrediction distribution:")

print(
    dns_production_df[
        "prediction"
    ].value_counts()
)

display(
    dns_production_df.head(20)
)

c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Production DNS results: 40953

Prediction distribution:
prediction
BENIGN        40361
DNS_THREAT      592
Name: count, dtype: int64


,prediction,model_score,decision_threshold,threat_class,severity,supporting_features
0,BENIGN,0.0886,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
1,BENIGN,0.0080,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
2,BENIGN,0.0030,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
3,BENIGN,0.0040,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
4,BENIGN,0.0063,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
5,BENIGN,0.0081,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
6,BENIGN,0.0068,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
7,BENIGN,0.0058,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
8,BENIGN,0.0075,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."
9,BENIGN,0.0074,0.93,DNS,LOW,"[{'feature': 'bytes_per_query', 'feature_value..."


In [9]:
# ============================================================
# STEP 207 — VERIFY PRODUCTION DNS DETECTOR
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

production_pred = (
    dns_production_df["prediction"]
    .eq("DNS_THREAT")
    .astype(int)
)

production_true = (
    dns_work["dns_target"]
    .astype(int)
)

print("==========================================")
print("PRODUCTION DNS DETECTOR VALIDATION")
print("==========================================")

print(
    "Total windows:",
    len(production_true)
)

print(
    "Actual DNS threats:",
    int(production_true.sum())
)

print(
    "Predicted DNS threats:",
    int(production_pred.sum())
)

print(
    "\nAccuracy:",
    f"{accuracy_score(production_true, production_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(production_true, production_pred, zero_division=0):.4f}"
)

print(
    "Recall:",
    f"{recall_score(production_true, production_pred, zero_division=0):.4f}"
)

print(
    "F1:",
    f"{f1_score(production_true, production_pred, zero_division=0):.4f}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        production_true,
        production_pred
    )
)

PRODUCTION DNS DETECTOR VALIDATION
Total windows: 40953
Actual DNS threats: 846
Predicted DNS threats: 592

Accuracy: 0.9921
Precision: 0.9409
Recall: 0.6584
F1: 0.7747

Confusion Matrix:
[[40072    35]
 [  289   557]]


In [10]:
# ============================================================
# STEP 208 — DNS ARTIFACT CHECK
# ============================================================

print("Model path:")
print(MODEL_PATH)

print("\nThreshold:")
print(THRESHOLD)

print("\nFeature count:")
print(len(FEATURES))

print("\nFeatures:")
for feature in FEATURES:
    print("-", feature)

Model path:
e:\CODEZILLA-SIH26145\models\dns_hgb.joblib

Threshold:
0.93

Feature count:
14

Features:
- dns_query_rate
- dns_unique_destinations
- dns_packet_concentration
- dns_byte_concentration
- dns_iat_cv
- query_rate_prev
- query_rate_roll3
- query_rate_roll6
- query_rate_std6
- query_rate_change
- query_rate_z6
- destination_change
- bytes_per_query
- packets_per_query


In [11]:
# ============================================================
# SAVE DNS PROCESSED FEATURES
# ============================================================

from pathlib import Path

processed_dir = (
    project_root
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

dns_artifact_path = (
    processed_dir
    / "dns_source_features.parquet"
)

dns_work.to_parquet(
    dns_artifact_path,
    index=False
)

print("✅ DNS features saved")
print("Path:", dns_artifact_path)
print("Shape:", dns_work.shape)
print("Features:", len(dns_ml_features))

✅ DNS features saved
Path: e:\CODEZILLA-SIH26145\data\processed\dns_source_features.parquet
Shape: (40953, 25)
Features: 14
